In [2]:
import pandas as pd

In [5]:
import dlt

In [7]:
import itertools
from itertools import islice
from dlt.sources.rest_api import rest_api_source

In [8]:
def openlibrary_source(query: str = "harry potter"):

    return rest_api_source({
        "client": {
            "base_url": "https://openlibrary.org",
        },
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "books",
                "endpoint": {
                    "path": "search.json",
                    "params": {
                        "q": query,
                        "limit": 100,
                    },
                    "data_selector": "docs",
                    "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
                },
            },
        ],
    })

In [9]:
pipeline = dlt.pipeline(
    pipeline_name="ol_demo",
    destination="duckdb",
    dataset_name="ol_data",
    progress="log" # logs the pipeline run (Optiona)
)

In [10]:
pipeline.run()

In [11]:
extract_info = pipeline.extract(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 205.94 MB (43.50%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.11s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 10754625.64/s
Memory usage: 206.57 MB (43.70%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 2.59s | Rate: 0.00/s
books: 400  | Time: 1.47s | Rate: 271.59/s
Memory usage: 206.69 MB (43.50%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 4.33s | Rate: 0.00/s
books: 600  | Time: 3.22s | Rate: 186.53/s
Memory usage: 207.32 MB (43.60%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 5.43s | Rate: 0

In [23]:
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()

Resources: ['books']
Tables: ['books']
Load ID: 1772435964.423191

Resource: books
rows extracted: 3759



In [24]:
normalize_info = pipeline.normalize()

------------------- Normalize rest_api in 1772435964.423191 --------------------
Files: 0/2 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 216.07 MB (46.30%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772435964.423191 --------------------
Files: 0/2 (0.0%) | Time: 0.00s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 216.07 MB (46.30%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772435964.423191 --------------------
Files: 10/2 (500.0%) | Time: 0.53s | Rate: 18.90/s
Items: 23083  | Time: 0.53s | Rate: 43670.98/s
Memory usage: 223.92 MB (46.30%) | CPU usage: 0.00%



In [26]:
load_id = normalize_info.loads_ids[-1]
m = normalize_info.metrics[load_id][0]

print("Load ID:", load_id)
print()
print("Tables created/updated:")
for table_name, tm in m["table_metrics"].items():
    # skip dlt internal tables to keep it beginner-friendly
    if table_name.startswith("_dlt"):
        continue
    print(f"  - {table_name}: {tm.items_count} rows")


Load ID: 1772435964.423191

Tables created/updated:
  - books: 3759 rows
  - books__author_key: 4637 rows
  - books__author_name: 4637 rows
  - books__ia: 3434 rows
  - books__ia_collection: 2735 rows
  - books__language: 3748 rows
  - books__id_standard_ebooks: 12 rows
  - books__id_librivox: 64 rows
  - books__id_project_gutenberg: 56 rows


In [37]:
# Display schema 
pipeline.default_schema

<dlt.Schema(name='rest_api', version=2, tables=['_dlt_version', '_dlt_loads', 'books', '_dlt_pipeline_state', 'books__author_key', 'books__author_name', 'books__ia', 'books__ia_collection', 'books__language', 'books__id_standard_ebooks', 'books__id_librivox', 'books__id_project_gutenberg'], version_hash='ZJIabaQJ9DAYgsR04wEVeXOgU80roBUfdvrR2YoBEyU=')>

In [29]:
pipeline.load()

---------------------- Load rest_api in 1772435964.423191 ----------------------
Jobs: 0/10 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 212.67 MB (46.30%) | CPU usage: 0.00%

---------------------- Load rest_api in 1772435964.423191 ----------------------
Jobs: 6/10 (60.0%) | Time: 1.19s | Rate: 5.04/s
Memory usage: 290.31 MB (47.20%) | CPU usage: 0.00%

---------------------- Load rest_api in 1772435964.423191 ----------------------
Jobs: 10/10 (100.0%) | Time: 1.94s | Rate: 5.15/s
Memory usage: 216.31 MB (46.30%) | CPU usage: 0.00%



LoadInfo(pipeline=<dlt.pipeline(pipeline_name='ol_demo', destination='duckdb', dataset_name='ol_data', default_schema_name='rest_api', schema_names=['rest_api'], first_run=False, dev_mode=False, is_active=True, pipelines_dir='/home/codespace/.dlt/pipelines', working_dir='/home/codespace/.dlt/pipelines/ol_demo')>, metrics={'1772435964.423191': [{'started_at': DateTime(2026, 3, 2, 7, 32, 56, 412836, tzinfo=Timezone('UTC')), 'finished_at': DateTime(2026, 3, 2, 7, 32, 58, 355695, tzinfo=Timezone('UTC')), 'job_metrics': {'books__ia.8d3fd2be15.insert_values.gz': LoadJobMetrics(job_id='books__ia.8d3fd2be15.insert_values.gz', file_path='/home/codespace/.dlt/pipelines/ol_demo/load/normalized/1772435964.423191/started_jobs/books__ia.8d3fd2be15.0.insert_values.gz', table_name='books__ia', started_at=DateTime(2026, 3, 2, 7, 32, 56, 705143, tzinfo=Timezone('UTC')), finished_at=DateTime(2026, 3, 2, 7, 32, 57, 331197, tzinfo=Timezone('UTC')), state='completed', remote_url=None, retry_count=0), 'books

In [30]:
ds = pipeline.dataset()

In [33]:
ds.tables

['books',
 'books__author_key',
 'books__author_name',
 'books__ia',
 'books__ia_collection',
 'books__language',
 'books__id_standard_ebooks',
 'books__id_librivox',
 'books__id_project_gutenberg',
 '_dlt_version',
 '_dlt_loads',
 '_dlt_pipeline_state']

In [36]:
df = ds.books__author_name.df()
df

,value,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,J. K. Rowling,QGKvo+6sadIMrw,0,1UdwOLcQjaxsqQ
1,J. K. Rowling,dkmRfmqjv1XkrQ,0,Gk3RQ3xSJslo5g
2,J. K. Rowling,ruLfPfwTzlUkMQ,0,pXCZC8MwtL9+3A
3,J. K. Rowling,4cllxkPHnGS4SQ,0,rC3j9GlWOMgHCw
4,J. K. Rowling,VafeDUIBEjyJdg,0,/x9aiZW9Peoyag
...,...,...,...,...
4632,"Field, John",pmnPEBvsLfwsQw,0,SvYbv2Yk62f4LQ
4633,Earl Warren,x1rmzUaX1gArGw,0,L+nTGP1BoUQnRQ
4634,Manju Jaidka,5Rnxh6xiFJTt5g,0,kaicrq42XihcCw
4635,Russell W. Davenport,EKgNyl+8QhNMVw,0,GyXDLWdhqa1jJQ
